## 🎯 Learning Objectives
* Understand the fundamental principles of Naive Bayes classification.
* Explain the 'naive' assumption and its implications in text classification.
* Implement a Naive Bayes model for text classification using scikit-learn.
* Interpret the results of a Naive Bayes classifier and evaluate its performance.
* Identify appropriate use cases and performance trade-offs for Naive Bayes in modern ML workflows.


## ML03-L06: Naive Bayes for Text Classification

Welcome to this lesson on Naive Bayes, a surprisingly powerful and efficient algorithm, especially for text classification tasks. Despite its simplicity and a rather strong assumption, it remains a cornerstone in many practical applications, serving as an excellent baseline or even a production-ready solution when speed and interpretability are paramount.

### The Core Idea: Probability and Bayes' Theorem

At its heart, Naive Bayes is a probabilistic classifier based on **Bayes' Theorem**. This theorem describes the probability of an event, based on prior knowledge of conditions that might be related to the event. In classification, we want to find the probability of a document belonging to a certain class, given the words present in it.

Bayes' Theorem states:

$$P(Class | Features) = \frac{P(Features | Class) \times P(Class)}{P(Features)}$$

Where:
*   $P(Class | Features)$ is the **posterior probability**: The probability of a document belonging to a specific class given its features (words).
*   $P(Features | Class)$ is the **likelihood**: The probability of observing these features given that the document belongs to that class.
*   $P(Class)$ is the **prior probability**: The overall probability of a document belonging to that class, regardless of its features.
*   $P(Features)$ is the **evidence**: The overall probability of observing these features, which acts as a normalizing constant.

Our goal is to find the class $C$ that maximizes $P(C | Features)$. Since $P(Features)$ is constant for all classes, we only need to maximize $P(Features | C) \times P(C)$.

### The 'Naive' Assumption

Here's where the "naive" part comes in. To simplify the calculation of $P(Features | Class)$, Naive Bayes makes a crucial assumption: **all features are conditionally independent of each other given the class.**

For text classification, this means that the presence of one word in a document is independent of the presence of any other word, given the document's class. For example, if we're classifying an email as "spam" or "not spam", the Naive Bayes model assumes that the probability of seeing the word "free" is independent of seeing the word "money", given that the email is spam. This is clearly a strong and often false assumption in real-world language, where words are highly dependent on each other (e.g., "machine" and "learning" often appear together).

Despite this unrealistic assumption, Naive Bayes often performs surprisingly well in practice. This is because, even if the individual probabilities are inaccurate, the *ranking* of probabilities for different classes can still be correct, leading to accurate classification.

### Naive Bayes for Text Classification: The Bag-of-Words Model

When applying Naive Bayes to text, we typically use the **Bag-of-Words (BoW)** model to represent documents. In BoW:

1.  **Tokenization**: The text is broken down into individual words (tokens).
2.  **Vocabulary Creation**: A unique list of all words across all documents is created.
3.  **Vectorization**: Each document is then represented as a vector where each dimension corresponds to a word in the vocabulary, and the value in that dimension is the count (or frequency) of that word in the document.

For text classification, the **Multinomial Naive Bayes** variant is most commonly used. It is suitable for features that represent counts or frequencies, which perfectly aligns with the Bag-of-Words model.

Let's walk through an example to see Naive Bayes in action for a simple text classification task.


In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# --- 1. Create a Sample Dataset ---
# For demonstration, we'll create a small, synthetic dataset of movie reviews
# and classify them as 'positive' or 'negative'.
data = {
    'review': [
        "This movie was fantastic! I loved every minute of it.",
        "Absolutely brilliant acting and a gripping plot.",
        "A total waste of time. So boring and predictable.",
        "I enjoyed the film, but the ending was a bit weak.",
        "Worst movie ever. Don't bother watching.",
        "Highly recommend! A masterpiece of cinema.",
        "Mediocre at best. I expected more.",
        "What a delightful surprise! Great story and characters.",
        "Could not finish it. Painfully slow.",
        "A must-see for all film enthusiasts. Truly inspiring."
    ],
    'sentiment': [
        'positive', 'positive', 'negative', 'positive', 'negative',
        'positive', 'negative', 'positive', 'negative', 'positive'
    ]
}
df = pd.DataFrame(data)

print("--- Sample Data ---")
print(df.head())
print("\n")

# --- 2. Prepare Data for Training ---
# Separate features (X) and target (y)
X = df['review']
y = df['sentiment']

# Split data into training and testing sets
# We use a small test size due to the tiny dataset, typically 0.2 or 0.3 is common.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}\n")

# --- 3. Feature Extraction: Convert text to numerical features (Bag-of-Words) ---
# CountVectorizer converts a collection of text documents to a matrix of token counts.
# It learns the vocabulary from the training data and transforms both train and test sets.
vectorizer = CountVectorizer(stop_words='english', lowercase=True)

# Fit the vectorizer on the training data and transform it
X_train_counts = vectorizer.fit_transform(X_train)

# Transform the test data using the *same* fitted vectorizer
X_test_counts = vectorizer.transform(X_test)

print("--- Feature Vectorization ---")
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")
print(f"Shape of training data (documents, features): {X_train_counts.shape}")
print(f"Shape of testing data (documents, features): {X_test_counts.shape}\n")

# Optional: Display a small part of the vectorized data and feature names
# print("Sample of vectorized training data (first 2 documents, first 10 features):\n", X_train_counts[:2, :10].toarray())
# print("Corresponding feature names (first 10):\n", vectorizer.get_feature_names_out()[:10])

# --- 4. Train the Naive Bayes Classifier ---
# Multinomial Naive Bayes is well-suited for discrete counts, like word counts in text.
model = MultinomialNB()

# Train the model using the vectorized training data and labels
model.fit(X_train_counts, y_train)

print("--- Model Training Complete ---\n")

# --- 5. Make Predictions ---
# Predict the sentiment for the test set reviews
y_pred = model.predict(X_test_counts)

print("--- Predictions ---")
for i in range(len(X_test)):
    print(f"Review: '{X_test.iloc[i][:50]}...' -> Actual: {y_test.iloc[i]}, Predicted: {y_pred[i]}")
print("\n")

# --- 6. Evaluate the Model ---
print("--- Model Evaluation ---")

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}\n")

# Display a detailed classification report (precision, recall, f1-score)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Display the confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# --- 7. Demonstrate with New Data ---
print("--- Predicting on New, Unseen Reviews ---")
new_reviews = [
    "This film was an absolute masterpiece, truly captivating.",
    "What a terrible experience, I regret watching it.",
    "It was okay, nothing special."
]

# Vectorize the new reviews using the *same* fitted vectorizer
new_reviews_counts = vectorizer.transform(new_reviews)

# Predict sentiment for the new reviews
new_predictions = model.predict(new_reviews_counts)

for review, prediction in zip(new_reviews, new_predictions):
    print(f"Review: '{review}' -> Predicted Sentiment: {prediction}")


### Interpreting the Output and Performance Trade-offs

In the code above, we performed a complete text classification workflow using Naive Bayes. Let's break down the output and discuss its implications:

1.  **Data Preparation**: We started with raw text reviews and their sentiments. The `train_test_split` function ensures we evaluate the model on unseen data, preventing overfitting.

2.  **Feature Extraction (`CountVectorizer`)**: This is a critical step. `CountVectorizer` transformed our human-readable text into numerical vectors. Each column in these vectors represents a unique word (token) from our training data's vocabulary, and the values are the counts of how many times that word appeared in a given review. We also used `stop_words='english'` to remove common words like "the", "a", "is" which often don't carry much sentiment, and `lowercase=True` for consistency.

3.  **Model Training (`MultinomialNB`)**: The `MultinomialNB` model was trained on these word count vectors. During training, it learns the prior probabilities of each class (e.g., how often 'positive' reviews appear) and the likelihoods of each word appearing given a class (e.g., how often "fantastic" appears in 'positive' reviews vs. 'negative' reviews).

4.  **Predictions and Evaluation**: We then used the trained model to predict sentiments for our test set. The evaluation metrics provide insight into the model's performance:
    *   **Accuracy**: The proportion of correctly classified instances. For our small dataset, an accuracy of `1.00` means all test samples were correctly classified. This is often achievable on very simple, clean, and small datasets, but rarely in real-world scenarios.
    *   **Classification Report**: This provides `precision`, `recall`, and `f1-score` for each class:
        *   **Precision**: Out of all reviews predicted as 'positive', how many were actually 'positive'? (True Positives / (True Positives + False Positives))
        *   **Recall**: Out of all actual 'positive' reviews, how many did the model correctly identify? (True Positives / (True Positives + False Negatives))
        *   **F1-Score**: The harmonic mean of precision and recall, offering a balance between the two.
    *   **Confusion Matrix**: A table showing the counts of true positive, true negative, false positive, and false negative predictions. For example, `[[2, 0], [0, 1]]` means 2 actual 'negative' reviews were predicted 'negative', and 1 actual 'positive' review was predicted 'positive', with no misclassifications.

### Performance Trade-offs and Use Cases

**Advantages of Naive Bayes:**

*   **Simplicity and Speed**: Naive Bayes models are incredibly fast to train and make predictions, even on large datasets. This makes them ideal for real-time applications or as a quick baseline.
*   **Good Performance on Text**: Despite its "naive" assumption, it often performs surprisingly well on text classification tasks, especially with high-dimensional data (many words).
*   **Requires Less Training Data**: Compared to more complex models, Naive Bayes can perform reasonably well with relatively smaller training datasets.
*   **Interpretability**: The model's probabilities can be inspected to understand which words contribute most to a particular class, offering a degree of interpretability.
*   **Scalability**: It scales linearly with the number of features and data points.

**Disadvantages of Naive Bayes:**

*   **Strong Independence Assumption**: The core assumption of feature independence is rarely true in real-world data, which can limit its performance on complex problems.
*   **Zero-Frequency Problem**: If a word appears in the test data that was not present in the training data for a particular class, the model will assign a zero probability to that class, leading to incorrect predictions. Smoothing techniques (like Laplace smoothing, which `MultinomialNB` implements by default with `alpha=1.0`) are used to mitigate this.
*   **Not a Good Estimator of Probabilities**: While it's good at classifying, the actual probability values it outputs might not be perfectly calibrated (i.e., a predicted probability of 0.8 might not truly mean an 80% chance).

**Typical Use Cases (Even in 2026):**

Even with the rise of large language models (LLMs) and deep learning, Naive Bayes remains highly relevant for:

*   **Spam Filtering**: A classic and still effective application.
*   **Sentiment Analysis**: For quick, high-throughput sentiment classification, especially on smaller datasets or when computational resources are limited.
*   **Document Classification**: Categorizing news articles, legal documents, or support tickets into predefined categories.
*   **Recommendation Systems**: Often used in collaborative filtering or content-based recommendation systems to classify user preferences.
*   **Baseline Model**: It's an excellent first model to try, providing a strong baseline against which more complex models can be compared. If Naive Bayes performs well, it might be sufficient, saving computational resources and development time.

Naive Bayes is a testament to the power of simple, probabilistic models. Understanding its mechanics and trade-offs is crucial for any ML engineer building robust and efficient systems.


### Resources for Further Learning

*   **Scikit-learn Documentation - Naive Bayes**: The official documentation for Naive Bayes implementations in Python's scikit-learn library. A great place to understand parameters and different variants.
    *   [https://scikit-learn.org/stable/modules/naive_bayes.html](https://scikit-learn.org/stable/modules/naive_bayes.html)
*   **Scikit-learn Documentation - Feature Extraction (Text)**: Learn more about `CountVectorizer`, `TfidfVectorizer`, and other text processing tools.
    *   [https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)
*   **Google AI Blog**: Explore articles on various ML topics, including foundational algorithms and their applications.
    *   [https://ai.googleblog.com/](https://ai.googleblog.com/)
*   **Hugging Face (for NLP context)**: While Naive Bayes is not a transformer, understanding modern NLP pipelines (tokenization, vectorization) from resources like Hugging Face can provide broader context for text processing.
    *   [https://huggingface.co/](https://huggingface.co/)
*   **Stanford CS229 Lecture Notes (Andrew Ng)**: For a deeper dive into the mathematical foundations of Naive Bayes and other probabilistic models.
    *   [http://cs229.stanford.edu/notes2020fall/cs229-notes1.pdf](http://cs229.stanford.edu/notes2020fall/cs229-notes1.pdf) (Refer to sections on Generative Learning Algorithms and Naive Bayes)
